## Prepare Outage Panel

In [1]:
# =============================================================================
# ESMI outage + district-CDD panel — SINGLE-CELL script
# =============================================================================
# Paste this whole thing into ONE Jupyter cell. Set the paths in CONFIG, run it.
# It returns `panel` (also written to OUT_PANEL_CSV).
#
# Pipeline:
#   Stage 1  Outage panel  : per (location, day) count minutes at 0 V
#            (outage_minutes) and minutes with any reading (observed_minutes,
#            out of 1440). NaN cells AND missing hour-rows both reduce coverage.
#            0 V is a clean outage flag (voltages are 0 or >=~180). 2014–2018
#            only (2019 export differs; excluded via EXCLUDE_FILE_SUBSTR).
#   Stage 2  District CDD  : ERA5-Land daily-mean t2m -> for EVERY GADM district,
#            area-weight-average the overlapping grid cells (weights = true
#            overlap area in an equal-area CRS; nearest-cell fallback for tiny
#            districts). Variables, all computed at the GRID-CELL level before
#            aggregation (so nonlinear quantities like sWBGT are averaged in
#            the right order), then area-weighted up to the district:
#              cdd_tmean    = max(district_mean_temp - base, 0)   (dry-bulb,
#                             district mean taken first -- kept for comparison)
#              cdd_cellmean = area-weighted mean of per-cell max(T - base, 0)
#              rh           = area-weighted mean of per-cell RH (Magnus/
#                             Bolton 1980), from t2m + d2m
#              cdd_swbgt    = area-weighted mean of per-cell max(sWBGT - base,
#                             0); sWBGT = 0.567*T + 0.393*e + 3.94
#                             (Willett & Sherwood 2012 / ABOM simplified WBGT)
#              cdd_dewpt    = area-weighted mean of per-cell max(Td - base, 0)
#              dewpt_c      = area-weighted mean of per-cell dewpoint (deg C)
#            The four humidity-derived variables (rh, cdd_swbgt, cdd_dewpt,
#            dewpt_c) require d2m; cdd_tmean/cdd_cellmean/tmean_c need only
#            t2m and are always computed. Computed for ALL districts so it
#            does NOT depend on the crosswalk.
#   Match    ESMI district strings -> GADM (NAME_1, NAME_2). Unmatched pairs are
#            written to OUT_UNMATCHED_CSV with fuzzy GADM-name suggestions; fix
#            them in an edited crosswalk CSV (CROSSWALK_CSV) and re-run.
#   Stage 3  Join outage x district-CDD; add day, month, year, district, state,
#            state_year, state_month (state x calendar month), state_yearmonth,
#            district_year, district_month (district x calendar month),
#            district_yearmonth.
#
# Re-running after a crosswalk edit: cached outage + CDD reload from CACHE_DIR,
# so only the match + join redo (seconds). Delete a cache file to force rebuild.
#
# Deps: pandas numpy xarray netCDF4(or h5netcdf) geopandas shapely pyproj pyarrow
# =============================================================================

import os
import re
import glob
import difflib
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

# --------------------------------------------------------------------------- #
# CONFIG  — EDIT THE PATHS                                                     #
# --------------------------------------------------------------------------- #
VOLTAGE_GLOBS = ["data/prayas_data/ESMI*voltage*.csv"]   # catches all year/month naming variants
EXCLUDE_FILE_SUBSTR = ["2019"]                        # drop 2019 files (different export)
LOCATION_CSV = "data/prayas_data/ESMI_location_information.csv"
ERA5_GLOB    = "data/prep_panel/prep_cdd/era5_land_daily/t2m_daily_mean_*.nc"
ERA5_D2M_GLOB = "data/prep_panel/prep_cdd/era5_land_daily/d2m_daily_mean_*.nc"
GADM_L2_PATH = "data/prep_panel/prep_cdd/gadm/gadm41_IND_2.shp"

CROSSWALK_CSV     = "data/prayas_out/esmi_gadm_crosswalk.csv"   # your EDITED crosswalk (see below)
OUT_PANEL_CSV     = "data/prayas_out/esmi_outage_cdd_panel.csv"
OUT_UNMATCHED_CSV = "data/prayas_out/esmi_unmatched_districts.csv"
OUT_SUGGEST_CSV   = "data/prayas_out/esmi_crosswalk_suggestions.csv"
CACHE_DIR         = "data/prayas_out/cache"
USE_CACHE         = False    # set False to force everything to rebuild

# Locations to drop outright (no district/state in the source metadata, so
# they can never join to a GADM district or get any climate exposure).
EXCLUDE_LOCATIONS = ["Masipidi-Hazaribagh"]  # 5 location-days, no district/state given

CDD_BASE_C      = 18.3
SWBGT_BASE_C    = 25.0     # base for simplified-WBGT CDD (Willett & Sherwood 2012 method)
DEWPOINT_BASE_C = 21.0     # base for dew-point CDD (moisture-only exposure)
MINUTES_IN_DAY = 1440
SHIFT_ERA5_TO_IST = False    # True = shift ERA5 +5:30 h before the daily mean
EQUAL_AREA_CRS = "EPSG:6933" # for true overlap-area weights (handles cos-lat exactly)
DATE_FORMATS = ["%d-%m-%Y", "%Y-%m-%d", "%d/%m/%Y", "%Y/%m/%d",
                 "%m/%d/%Y", "%d-%b-%Y", "%d %b %Y", "%d-%m-%y"]

# ESMI state string -> single GADM NAME_1 (dual labels resolved)
STATE_NORMALIZE = {
    "andhra pradesh/telangana": "Telangana",   # Hyderabad & Rangareddy are Telangana
    "telangana": "Telangana",
    "andhra pradesh": "Andhra Pradesh",
    "uttaranchal/uttarakhand": "Uttarakhand",
    "chandigarh (ut)": "Chandigarh",
    "delhi": "NCT of Delhi",
}

# (norm_state, norm_district) -> GADM NAME_2. Best-guess seeds; VERIFY against
# your GADM. The edited CROSSWALK_CSV (below) overrides anything here.
DISTRICT_CROSSWALK = {
    ("Bihar", "sasaram"):                        "Rohtas",       # town -> its district
    ("Goa", "salcete"):                          "South Goa",    # taluka -> district
    ("Karnataka", "bengaluru urban"):            "Bangalore",
    ("Telangana", "rangareddy"):                 "Ranga Reddy",
    ("Assam", "kamrup metro"):                   "Kamrup Metropolitan",
    ("Andhra Pradesh", "anantpur"):              "Anantapur",
    ("Haryana", "gurgaon"):                      "Gurgaon",
    ("Karnataka", "belgaum"):                    "Belgaum",
    ("Karnataka", "bijapur"):                    "Bijapur",
    ("Uttar Pradesh", "gautam buddh nagar"):     "Gautam Buddha Nagar",
    ("Uttar Pradesh", "kanpur"):                 "Kanpur Nagar",
    ("West Bengal", "north 24 parganas"):        "North 24 Parganas",
    ("Maharashtra", "mumbai suburban district"): "Mumbai Suburban",
    ("Maharashtra", "mumbai"):                   "Mumbai City",
    ("Punjab", "sahibzada ajit singh nagar"):    "Sahibzada Ajit Singh Nagar",
    ("Odisha", "khordha"):                       "Khordha",
    ("Odisha", "kendujhar"):                     "Kendujhar",
    ("Uttar Pradesh", "faizabad"):               "Faizabad",
    ("Madhya Pradesh", "hoshangabad"):           "Hoshangabad",
    ("NCT of Delhi", "north east delhi"): "West", ("NCT of Delhi", "south west delhi"): "West",
    ("NCT of Delhi", "north west delhi"): "West", ("NCT of Delhi", "central delhi"): "West",
    ("NCT of Delhi", "east delhi"): "West", ("NCT of Delhi", "south delhi"): "West",
    ("NCT of Delhi", "west delhi"): "West",
}
# EDITED CROSSWALK CSV (CROSSWALK_CSV): same columns as OUT_UNMATCHED_CSV —
# State, District, state_norm, district_gadm. Take the unmatched report, set the
# last two columns to the EXACT GADM NAME_1 / NAME_2 (the suggestions printed at
# the end give you these), and save it under CROSSWALK_CSV (a DIFFERENT file, so
# it's never overwritten). Blank district_gadm rows are skipped, so you can fill
# it in incrementally. This CSV wins over DISTRICT_CROSSWALK above.


# --------------------------------------------------------------------------- #
# HELPERS                                                                      #
# --------------------------------------------------------------------------- #
def _norm(s):
    if not isinstance(s, str):
        return ""
    s = re.sub(r"[\.\-_]", " ", s.strip().lower())
    return re.sub(r"\s+", " ", s).strip()


def normalize_state(raw):
    return STATE_NORMALIZE.get(_norm(raw), raw.strip())


def add_gadm_key(gadm):
    gadm = gadm.copy()
    gadm["gadm_key"] = gadm["NAME_1"].astype(str) + "||" + gadm["NAME_2"].astype(str)
    return gadm


def sat_vp_hpa(t_degc):
    """Magnus saturation vapour pressure over water, hPa (Bolton 1980)."""
    return 6.112 * np.exp(17.67 * t_degc / (t_degc + 243.5))


def load_crosswalk_csv(path):
    """(raw State, raw District) -> (GADM NAME_1, NAME_2) from the edited CSV."""
    if not path or not os.path.exists(path):
        return {}
    cw = pd.read_csv(path)
    cw.columns = [c.strip() for c in cw.columns]
    need = {"State", "District", "state_norm", "district_gadm"}
    if not need.issubset(cw.columns):
        raise ValueError(f"{path} must contain columns {sorted(need)}")
    out = {}
    for row in cw.itertuples(index=False):
        dg, sn = getattr(row, "district_gadm"), getattr(row, "state_norm")
        if isinstance(dg, str) and dg.strip() and isinstance(sn, str) and sn.strip():
            out[(str(row.State).strip(), str(row.District).strip())] = (sn.strip(), dg.strip())
    return out


# --------------------------------------------------------------------------- #
# STAGE 1 — OUTAGE PANEL                                                       #
# --------------------------------------------------------------------------- #
def parse_dates_robust(raw, fname):
    """Parse Date with whichever explicit format parses the most rows."""
    s = raw.astype(str).str.strip()
    best, best_ok, best_fmt = None, -1, None
    for fmt in DATE_FORMATS:
        d = pd.to_datetime(s, format=fmt, errors="coerce")
        ok = int(d.notna().sum())
        if ok > best_ok:
            best, best_ok, best_fmt = d, ok, fmt
    frac = best_ok / len(s) if len(s) else 0.0
    print(f"    dates via {best_fmt!r}: {best_ok:,}/{len(s):,} ({100*frac:.1f}%)")
    if frac < 0.999:
        bad = s[best.isna() & (s.str.lower() != "nan")].value_counts().head(6)
        if len(bad):
            print(f"    unparsed examples: {dict(bad)}")
    return best


def build_outage_panel(voltage_globs):
    files = sorted({f for pat in voltage_globs for f in glob.glob(pat)})
    dropped = [f for f in files if any(x in os.path.basename(f) for x in EXCLUDE_FILE_SUBSTR)]
    files = [f for f in files if f not in dropped]
    if dropped:
        print(f"[stage1] excluding: {[os.path.basename(f) for f in dropped]}")
    if not files:
        raise FileNotFoundError(f"No voltage CSVs matched {voltage_globs}")
    print(f"[stage1] {len(files)} voltage files")

    frames = []
    for f in files:
        df = pd.read_csv(f, low_memory=False)
        df.columns = [c.strip() for c in df.columns]
        min_cols = [c for c in df.columns if re.fullmatch(r"Min \d+", c)]
        if not min_cols:
            print(f"  !! skipping (no Min columns): {f}")
            continue
        print(f"  read {os.path.basename(f)}: {len(df):,} rows")
        vals = df[min_cols].apply(pd.to_numeric, errors="coerce")
        out = pd.DataFrame({
            "Location name": df["Location name"].astype(str).str.strip(),
            "date": parse_dates_robust(df["Date"], os.path.basename(f)),
            "Hour": pd.to_numeric(df["Hour"], errors="coerce").astype("Int64"),
            "row_outage": (vals == 0).sum(axis=1).astype("int64"),       # minutes at 0 V
            "row_observed": vals.notna().sum(axis=1).astype("int64"),    # minutes with a reading
        })
        n_bad = int(out["date"].isna().sum())
        if n_bad:
            print(f"    -> dropping {n_bad:,} unparseable-date rows")
        out = out.dropna(subset=["date"])
        frames.append(out)

    long = pd.concat(frames, ignore_index=True)
    long = long.drop_duplicates(subset=["Location name", "date", "Hour"], keep="first")
    panel = (long.groupby(["Location name", "date"], as_index=False)
                 .agg(outage_minutes=("row_outage", "sum"),
                      observed_minutes=("row_observed", "sum")))
    panel["observed_minutes"] = panel["observed_minutes"].clip(upper=MINUTES_IN_DAY)
    print(f"[stage1] outage panel: {len(panel):,} location-days")
    return panel


# --------------------------------------------------------------------------- #
# STAGE 2 — DISTRICT DAILY CDD FROM ERA5-LAND                                  #
# --------------------------------------------------------------------------- #
def load_era5_daily_mean(era5_glob, var_candidates=("t2m", "2t", "tas", "t2m_mean")):
    """Open a set of ERA5-Land daily-statistics files -> daily-mean DataArray
    in degC, dims harmonised to (time, latitude/lat, longitude/lon).
    var_candidates lets this be reused for any variable (t2m, d2m, ...)."""
    files = sorted(glob.glob(era5_glob))
    if not files:
        raise FileNotFoundError(f"No ERA5 files matched {era5_glob}")
    print(f"[stage2] {len(files)} ERA5-Land files ({era5_glob})")
    ds = xr.open_mfdataset(files, combine="by_coords")

    tvar = next((c for c in var_candidates if c in ds.variables), None)
    if tvar is None:
        tvar = next((v for v in ds.data_vars if ds[v].ndim >= 3), None)
    if tvar is None:
        raise ValueError(f"No matching variable found in {era5_glob} "
                          f"(looked for {var_candidates})")
    da = ds[tvar]
    print(f"[stage2] var = {tvar!r}  long_name="
          f"{da.attrs.get('long_name', da.attrs.get('standard_name', '?'))!r} "
          f"units={da.attrs.get('units', '?')!r}  <- confirm this is a MEAN")

    lat_name = "latitude" if "latitude" in da.coords else "lat"
    lon_name = "longitude" if "longitude" in da.coords else "lon"
    time_name = "valid_time" if "valid_time" in da.coords else "time"
    da = da.rename({time_name: "time"})

    if SHIFT_ERA5_TO_IST:
        da = da.assign_coords(time=da["time"] + np.timedelta64(330, "m"))

    t = pd.to_datetime(da["time"].values)
    if len(t) > 1:
        step_h = np.median(np.diff(t)).astype("timedelta64[h]").astype(int)
        if step_h < 24:
            print(f"[stage2] sub-daily ({step_h} h) -> resampling to daily mean")
            da = da.resample(time="1D").mean()
    da = da.assign_coords(time=pd.to_datetime(da["time"].values).normalize())

    units = str(da.attrs.get("units", "")).lower()
    if units in ("k", "kelvin") or float(da.isel(time=0).max().compute()) > 100:
        da = da - 273.15
        da.attrs["units"] = "degC"
        print("[stage2] converted Kelvin -> degC")
    return da, lat_name, lon_name


def build_cell_grid(da, lat_name, lon_name):
    lats, lons = da[lat_name].values, da[lon_name].values
    dlat = float(np.median(np.abs(np.diff(lats))))
    dlon = float(np.median(np.abs(np.diff(lons))))
    geoms, ilat, ilon, clat, clon = [], [], [], [], []
    for i, la in enumerate(lats):
        for j, lo in enumerate(lons):
            geoms.append(box(lo - dlon/2, la - dlat/2, lo + dlon/2, la + dlat/2))
            ilat.append(i); ilon.append(j); clat.append(la); clon.append(lo)
    cells = gpd.GeoDataFrame({"ilat": ilat, "ilon": ilon, "clat": clat, "clon": clon},
                              geometry=geoms, crs="EPSG:4326")
    return cells, dlat, dlon


def district_daily_cdd(da, lat_name, lon_name, cells, gadm_districts, da_td=None):
    """Area-weighted district-day exposures.

    All quantities are computed at the GRID-CELL level first, then
    area-weight-averaged across the cells overlapping each district -- the
    same order of operations as the population-weighted state panel script,
    just with overlap-area weights instead of population weights.

    da_td (optional): dewpoint DataArray, same grid as da, already time-
    reindexed onto da's time axis (missing dates -> NaN, not dropped). When
    given, also returns dewpt_c, rh, cdd_swbgt, cdd_dewpt.
    """
    cells_ea = cells.to_crs(EQUAL_AREA_CRS)
    T = da.transpose("time", lat_name, lon_name).values
    times = pd.to_datetime(da["time"].values).normalize()

    TD = None
    if da_td is not None:
        TD = da_td.transpose("time", lat_name, lon_name).values
        assert TD.shape == T.shape, "t2m/d2m arrays don't line up -- check reindex"

    # static land mask: a cell counts as usable if it has data at ANY timestep
    cell_has_data = ~np.isnan(T).all(axis=0)   # (nlat, nlon)
    if TD is not None:
        cell_has_data &= ~np.isnan(TD).all(axis=0)

    records, n_fallback, n_partial = [], 0, 0
    for _, drow in gadm_districts.iterrows():
        poly = drow.geometry
        cand = cells.iloc[list(cells.sindex.query(poly, predicate="intersects"))]
        weights = None
        if len(cand):
            poly_ea = gpd.GeoSeries([poly], crs="EPSG:4326").to_crs(EQUAL_AREA_CRS).iloc[0]
            w = cells_ea.loc[cand.index].geometry.intersection(poly_ea).area.values
            keep = w > 0
            cand, w = cand[keep], w[keep]
            if len(cand):
                weights = w

        if weights is not None and len(cand):
            valid = cell_has_data[cand["ilat"].values, cand["ilon"].values]
            if valid.any():
                if not valid.all():
                    n_partial += 1
                cand, weights = cand[valid], weights[valid]
            else:
                weights = None   # everything overlapping is masked -> fall through to fallback

        if weights is None or len(cand) == 0:            # tiny/fully-masked district -> nearest VALID cell
            c = poly.centroid
            d2 = (cells["clon"].values - c.x)**2 + (cells["clat"].values - c.y)**2
            d2 = np.where(cell_has_data[cells["ilat"].values, cells["ilon"].values], d2, np.inf)
            cand = cells.iloc[[int(np.argmin(d2))]]
            weights = np.array([1.0]); n_fallback += 1

        w = weights / weights.sum()
        ilat, ilon = cand["ilat"].values, cand["ilon"].values
        series = T[:, ilat, ilon]                          # (nt, ncell), no NaNs left
        tmean = series @ w

        out = {
            "gadm_key": drow["gadm_key"], "date": times, "tmean_c": tmean,
            "cdd_tmean": np.maximum(tmean - CDD_BASE_C, 0.0),
            "cdd_cellmean": np.maximum(series - CDD_BASE_C, 0.0) @ w,
        }

        if TD is not None:
            td_series = TD[:, ilat, ilon]                   # (nt, ncell) dewpoint, deg C
            e_cell = sat_vp_hpa(td_series)                    # actual vapour pressure, hPa
            es_cell = sat_vp_hpa(series)                      # saturation VP at cell air temp
            swbgt_cell = 0.567 * series + 0.393 * e_cell + 3.94   # ABOM simplified WBGT
            rh_cell = np.clip(100.0 * e_cell / es_cell, 0.0, 100.0)

            # NaN-safe weighted sums: where a given day/cell has no d2m value,
            # exclude that cell from THAT day's weighted mean rather than
            # letting one NaN cell poison the whole row.
            def wmean_nan(x):
                valid = np.isfinite(x)
                wm = np.where(valid, w, 0.0)
                denom = wm.sum(axis=1)
                num = np.nan_to_num(x * wm, nan=0.0).sum(axis=1)
                return np.where(denom > 0, num / denom, np.nan)

            out["dewpt_c"] = wmean_nan(td_series)
            out["rh"] = wmean_nan(rh_cell)
            out["cdd_swbgt"] = wmean_nan(np.maximum(swbgt_cell - SWBGT_BASE_C, 0.0))
            out["cdd_dewpt"] = wmean_nan(np.maximum(td_series - DEWPOINT_BASE_C, 0.0))

        records.append(pd.DataFrame(out))

    if n_partial:
        print(f"[stage2] dropped ocean/masked cells (renormalized weights) for {n_partial} district(s)")
    if n_fallback:
        print(f"[stage2] nearest-valid-cell fallback for {n_fallback} district(s)")
    out_df = pd.concat(records, ignore_index=True)
    print(f"[stage2] district-day CDD rows: {len(out_df):,}")
    return out_df


# --------------------------------------------------------------------------- #
# DISTRICT MATCHING + SUGGESTIONS                                             #
# --------------------------------------------------------------------------- #
def match_districts(loc, gadm):
    gadm = add_gadm_key(gadm)
    gadm["NAME_1"] = gadm["NAME_1"].astype(str)
    gadm["NAME_2"] = gadm["NAME_2"].astype(str)
    k1 = gadm["NAME_1"].map(_norm); k2 = gadm["NAME_2"].map(_norm)
    gadm_lookup = dict(zip(zip(k1, k2), zip(gadm["NAME_1"], gadm["NAME_2"])))

    csv_override = load_crosswalk_csv(CROSSWALK_CSV)
    if csv_override:
        print(f"[match] {len(csv_override)} manual crosswalk row(s) from {CROSSWALK_CSV}")

    def resolve(state_raw, dist_raw):
        ov = csv_override.get((str(state_raw).strip(), str(dist_raw).strip()))
        if ov is not None:
            key_ov = (_norm(ov[0]), _norm(ov[1]))
            if key_ov in gadm_lookup:
                g1, g2 = gadm_lookup[key_ov]; return g1, g2, True
            return ov[0], ov[1], False
        state = normalize_state(state_raw)
        cw = DISTRICT_CROSSWALK.get((state, _norm(dist_raw)))
        target = cw if cw is not None else dist_raw.strip()
        k = (_norm(state), _norm(target))
        if k in gadm_lookup:
            g1, g2 = gadm_lookup[k]; return g1, g2, True
        return state, target, False

    recs = loc[["State", "District"]].drop_duplicates()
    resolved = {(s, d): resolve(s, d) for s, d in recs.itertuples(index=False)}
    loc = loc.copy()
    loc["state_norm"]    = loc.apply(lambda r: resolved[(r.State, r.District)][0], axis=1)
    loc["district_gadm"] = loc.apply(lambda r: resolved[(r.State, r.District)][1], axis=1)
    loc["matched"]       = loc.apply(lambda r: resolved[(r.State, r.District)][2], axis=1)
    loc["gadm_key"]      = loc["state_norm"] + "||" + loc["district_gadm"]

    unmatched = (loc.loc[~loc["matched"], ["State", "District", "state_norm", "district_gadm"]]
                    .drop_duplicates().sort_values(["State", "District"]))
    print(f"[match] {loc['matched'].sum()} location rows matched, "
          f"{len(unmatched)} distinct unmatched (state,district) pair(s)")
    return loc, unmatched


def suggest_crosswalk(gadm, loc, unmatched):
    """Fuzzy GADM-name suggestions for each unmatched pair, busiest first."""
    cols = ["n_locations", "State", "District", "state_norm", "level", "suggestions", "best"]
    if unmatched.empty:
        return pd.DataFrame(columns=cols)
    gadm = gadm.copy()
    gadm["NAME_1"] = gadm["NAME_1"].astype(str); gadm["NAME_2"] = gadm["NAME_2"].astype(str)
    states = sorted(gadm["NAME_1"].unique())
    by_state = {s: sorted(gadm.loc[gadm["NAME_1"] == s, "NAME_2"].unique()) for s in states}
    counts = loc.groupby(["State", "District"]).size().rename("n_locations").reset_index()
    um = unmatched.merge(counts, on=["State", "District"], how="left") \
                  .sort_values("n_locations", ascending=False)
    rows = []
    for r in um.itertuples(index=False):
        sn = r.state_norm; n = getattr(r, "n_locations", None)
        if sn not in by_state:
            near = difflib.get_close_matches(sn, states, n=3, cutoff=0.3)
            rows.append(dict(n_locations=n, State=r.State, District=r.District, state_norm=sn,
                              level="STATE_NOT_IN_GADM", suggestions=near, best=(near[0] if near else "")))
            continue
        nm = {_norm(c): c for c in by_state[sn]}
        near = difflib.get_close_matches(_norm(r.District), list(nm), n=4, cutoff=0.3)
        sug = [nm[k] for k in near]
        rows.append(dict(n_locations=n, State=r.State, District=r.District, state_norm=sn,
                          level=("district" if sug else "no_candidate"),
                          suggestions=sug, best=(sug[0] if sug else "")))
    return pd.DataFrame(rows, columns=cols)




# --------------------------------------------------------------------------- #
# STAGE 3 — ASSEMBLE                                                           #
# --------------------------------------------------------------------------- #
def assemble(outage, loc, cdd):
    loc_small = loc[["Location name", "state_norm", "district_gadm", "gadm_key"]].drop_duplicates()
    panel = outage.merge(loc_small, on="Location name", how="left") \
                  .merge(cdd, on=["gadm_key", "date"], how="left")
    panel["day"]   = panel["date"].dt.day
    panel["month"] = panel["date"].dt.month
    panel["year"]  = panel["date"].dt.year
    panel = panel.rename(columns={"state_norm": "state", "district_gadm": "district"})
    panel["state_year"]         = panel["state"] + "_" + panel["year"].astype(str)
    panel["state_month"]        = panel["state"] + "_" + panel["month"].astype(str)           # x calendar month
    panel["state_yearmonth"]    = panel["state"] + "_" + panel["date"].dt.strftime("%Y-%m")   # x year-month
    panel["district_year"]      = panel["district"] + "_" + panel["year"].astype(str)
    panel["district_month"]     = panel["district"] + "_" + panel["month"].astype(str)         # x calendar month
    panel["district_yearmonth"] = panel["district"] + "_" + panel["date"].dt.strftime("%Y-%m") # x year-month
    cols = ["Location name", "date", "day", "month", "year",
            "district", "state", "state_year", "state_month", "state_yearmonth",
            "district_year", "district_month", "district_yearmonth",
            "outage_minutes", "observed_minutes",
            "tmean_c", "cdd_tmean", "cdd_cellmean",
            "dewpt_c", "rh", "cdd_swbgt", "cdd_dewpt"]
    cols = [c for c in cols if c in panel.columns]
    return panel[cols].sort_values(["Location name", "date"]).reset_index(drop=True)


# --------------------------------------------------------------------------- #
# RUN                                                                          #
# --------------------------------------------------------------------------- #
def run():
    os.makedirs(CACHE_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(OUT_PANEL_CSV) or ".", exist_ok=True)

    # Stage 1 (cached)
    outage_cache = os.path.join(CACHE_DIR, "outage_panel.parquet")
    if USE_CACHE and os.path.exists(outage_cache):
        outage = pd.read_parquet(outage_cache); print(f"[cache] outage {outage.shape}")
    else:
        outage = build_outage_panel(VOLTAGE_GLOBS); outage.to_parquet(outage_cache)

    # Drop locations with no district/state in the source metadata (applied
    # here, after cache load, so it takes effect even with USE_CACHE=True and
    # doesn't require a cache rebuild).
    if EXCLUDE_LOCATIONS:
        n_before = len(outage)
        outage = outage[~outage["Location name"].isin(EXCLUDE_LOCATIONS)]
        print(f"[stage1] dropped {n_before - len(outage):,} location-days for "
              f"excluded locations: {EXCLUDE_LOCATIONS}")

    # Locations + GADM (dissolve any duplicate district keys)
    loc = pd.read_csv(LOCATION_CSV); loc.columns = [c.strip() for c in loc.columns]
    loc["Location name"] = loc["Location name"].astype(str).str.strip()
    gadm = add_gadm_key(gpd.read_file(GADM_L2_PATH))
    if gadm["gadm_key"].duplicated().any():
        gadm = gadm.dissolve(by="gadm_key", as_index=False)

    # Stage 2: CDD (+ RH / sWBGT-CDD / dewpoint-CDD) for ALL districts
    # (cached; crosswalk-independent). New cache filename so an old,
    # t2m-only cache from before this change isn't silently reused without
    # the humidity columns.
    cdd_cache = os.path.join(CACHE_DIR, "cdd_all_districts_humid.parquet")
    if USE_CACHE and os.path.exists(cdd_cache):
        cdd_all = pd.read_parquet(cdd_cache); print(f"[cache] cdd {cdd_all.shape}")
    else:
        da_t, lat_name, lon_name = load_era5_daily_mean(ERA5_GLOB)
        da_td, lat_name_d, lon_name_d = load_era5_daily_mean(
            ERA5_D2M_GLOB, var_candidates=("d2m", "2d", "tdps"))
        assert (lat_name, lon_name) == (lat_name_d, lon_name_d), \
            "t2m/d2m use different coordinate names -- check the files"
        assert np.allclose(da_t[lat_name].values, da_td[lat_name_d].values) and \
               np.allclose(da_t[lon_name].values, da_td[lon_name_d].values), \
               "t2m/d2m grids don't match spatially"
        # Keep ALL t2m dates; reindex (not inner-join) so a d2m gap only nulls
        # the humidity columns for that date instead of dropping cdd_tmean too.
        da_td = da_td.reindex(time=da_t["time"])

        cells, _, _ = build_cell_grid(da_t, lat_name, lon_name)
        cdd_all = district_daily_cdd(da_t, lat_name, lon_name, cells, gadm, da_td=da_td)
        cdd_all.to_parquet(cdd_cache)

    # Match ESMI -> GADM, report + suggest
    loc_keyed, unmatched = match_districts(loc, gadm)
    unmatched.to_csv(OUT_UNMATCHED_CSV, index=False)
    if len(unmatched):
        sug = suggest_crosswalk(gadm, loc, unmatched)
        sug.to_csv(OUT_SUGGEST_CSV, index=False)
        print(f"\n[crosswalk] {len(unmatched)} unmatched — suggested GADM names "
              f"(also -> {OUT_SUGGEST_CSV}):")
        with pd.option_context("display.max_rows", None, "display.width", 200):
            print(sug.to_string(index=False))
        print(f"\n-> Fix state_norm/district_gadm in {OUT_UNMATCHED_CSV}, "
              f"save as {CROSSWALK_CSV}, re-run this cell.")

    # Stage 3: join + write
    panel = assemble(outage, loc_keyed, cdd_all)
    panel.to_csv(OUT_PANEL_CSV, index=False)
    n_nocdd = int(panel["cdd_tmean"].isna().sum())
    print(f"\n[done] {len(panel):,} rows -> {OUT_PANEL_CSV} | no dry-bulb-CDD rows: {n_nocdd:,}")
    if n_nocdd:
        miss = (panel[panel["cdd_tmean"].isna()]
                    .groupby(["state", "district"]).size().sort_values(ascending=False))
        print("no dry-bulb-CDD by district (top 10 — crosswalk gap vs date gap):")
        print(miss.head(10).to_string())

    for v in ("rh", "cdd_swbgt", "cdd_dewpt"):
        if v in panel.columns:
            n_missing = int(panel[v].isna().sum())
            if n_missing:
                print(f"no-{v} rows (d2m still missing for these dates): {n_missing:,}")

    return panel


# Runs on paste. After the first run, edit the crosswalk CSV and just re-run the
# cell — cached outage + CDD reload, only the match + join redo.
panel = run()

[stage1] excluding: ['ESMI voltage data 2019 Jan-June.csv', 'ESMI voltage data 2019 July-Dec.csv']
[stage1] 10 voltage files
  read ESMI minute-wise voltage data 2014.csv: 58,976 rows
    dates via '%d-%m-%Y': 58,976/58,976 (100.0%)
  read ESMI minute-wise voltage data 2015.csv: 766,649 rows
    dates via '%d-%m-%Y': 766,649/766,649 (100.0%)
  read ESMI minute-wise voltage data 2016 Jan-June.csv: 778,400 rows
    dates via '%d-%m-%Y': 778,400/778,400 (100.0%)
  read ESMI minute-wise voltage data 2016 July-Dec.csv: 874,773 rows
    dates via '%d-%m-%Y': 874,773/874,773 (100.0%)
  read ESMI minute-wise voltage data 2017 Jan-April.csv: 696,857 rows
    dates via '%d-%m-%Y': 696,857/696,857 (100.0%)
  read ESMI minute-wise voltage data 2017 May-Aug.csv: 681,422 rows
    dates via '%d-%m-%Y': 681,422/681,422 (100.0%)
  read ESMI minute-wise voltage data 2017 Sept-Dec.csv: 632,196 rows
    dates via '%d-%m-%Y': 632,196/632,196 (100.0%)
  read ESMI minute-wise voltage data 2018 Jan-May.csv: 8

## Check for NaN values

In [ ]:
import pandas as pd

OUT_PANEL_CSV = "data/prayas_out/esmi_outage_cdd_panel.csv"   # adjust if needed

panel = pd.read_csv(OUT_PANEL_CSV)

n_rows = len(panel)
print(f"Panel: {n_rows:,} rows, {panel.shape[1]} columns\n")

na_counts = panel.isna().sum()
na_counts = na_counts[na_counts > 0].sort_values(ascending=False)

if na_counts.empty:
    print("No NaNs anywhere in the panel.")
else:
    print("NaN counts by column:")
    for col, n in na_counts.items():
        print(f"  {col:20s} {n:6,d}  ({100*n/n_rows:5.2f}%)")

    # Rows with at least one NaN, broken down by location -- helps spot
    # whether missingness is concentrated in a few locations/dates or spread
    # thin across the panel.
    any_na = panel[panel.isna().any(axis=1)]
    print(f"\n{len(any_na):,} rows have at least one NaN "
          f"({100*len(any_na)/n_rows:.2f}% of the panel)")

    if "Location name" in panel.columns and len(any_na):
        print("\nTop locations by NaN row count:")
        print(any_na["Location name"].value_counts().head(10).to_string())

Panel: 278,225 rows, 22 columns

No NaNs anywhere in the panel.
